# Highway Network na CIFAR-10

[Highway Networks](https://arxiv.org/abs/1505.00387) (Srivastava, Greff, Schmidhuber, 2015)

In [ ]:
import io, os, time, urllib.request
import numpy as np
import pyarrow.parquet as pq
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
EPOCHS = 100
BATCH_SIZE = 128
LR_DECAY = 0.1
LR_MILESTONES = (0.5, 0.75)

SEARCH_TRIALS = 20
SEARCH_EPOCHS = 5
SEARCH_VAL = 5000
LR_RANGE = (0.01, 0.2)
MOMENTUM_RANGE = (0.8, 0.95)
GATE_BIAS_RANGE = (-4.0, -1.0)

SEED = 0

torch.manual_seed(SEED)
DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True
print("GPU:", torch.cuda.get_device_name(0))

## Dane

CIFAR-10: 50 000 obrazów treningowych i 10 000 testowych, 32x32 RGB, 10 klas.

In [ ]:
def load_split(split):
    url = f"https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/{split}-00000-of-00001.parquet"
    path = f"cifar10_{split}.parquet"
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    table = pq.read_table(path).to_pydict()
    images = np.stack([np.asarray(Image.open(io.BytesIO(im["bytes"])).convert("RGB")) for im in table["img"]])
    labels = np.array(table["label"])
    return images, labels

x_train, y_train = load_split("train")
x_test, y_test = load_split("test")
print(x_train.shape, x_test.shape)

In [ ]:
x_train = torch.tensor(x_train).permute(0, 3, 1, 2).float() / 255  # [N, 32, 32, 3] -> [N, 3, 32, 32], wartości 0-1
x_test = torch.tensor(x_test).permute(0, 3, 1, 2).float() / 255
# normalizacja każdego kanału RGB: inicjalizacja He zakłada wejście o średniej 0 i wariancji 1
mean = x_train.mean(dim=(0, 2, 3), keepdim=True)
std = x_train.std(dim=(0, 2, 3), keepdim=True)
x_train, x_test = (x_train - mean) / std, (x_test - mean) / std
y_train, y_test = torch.tensor(y_train), torch.tensor(y_test)

# augmentacja jak w paperze ("random translations"): losowe przesunięcie o maks. 4 piksele w każdą stronę,
# więc w każdej epoce sieć widzi trochę inne obrazy i trudniej jej zapamiętać zbiór treningowy
shift = T.RandomCrop(32, padding=4)

class Shifted(torch.utils.data.Dataset):
    def __init__(self, x, y): self.x, self.y = x, y
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return shift(self.x[i]), self.y[i]

def loader(x, y, train):
    ds = Shifted(x, y) if train else torch.utils.data.TensorDataset(x, y)
    return torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=train, drop_last=train,
                                       num_workers=2, pin_memory=True, persistent_workers=True)

## Warstwa highway

$$y = H(x)\cdot T(x) + x\cdot\big(1 - T(x)\big)$$

$$H(x) = \mathrm{ReLU}(W_H * x + b_H), \qquad T(x) = \sigma(W_T * x + b_T)$$

$T(x)$ [1] decyduje, jaka część wyjścia to przetworzone $H(x)$, a jaka niezmienione $x$ [2]. $x$ i $H(x)$ muszą mieć ten sam rozmiar [3].

[1] $*$ to konwolucja: filtr 3x3 przesuwa się po obrazie i w każdym miejscu mnoży piksele pod sobą przez swoje wagi i sumuje (po wszystkich kanałach), co daje jeden piksel wyjścia. $\sigma$ to sigmoida, więc $T(x)$ ma wartości od 0 do 1, osobno dla każdego kanału i piksela.

[2] Dla $T = 0$ warstwa przepuszcza $x$ bez zmian, razem z gradientem. Dzięki temu gradient nie zanika nawet w głębokiej sieci.

[3] Stąd padding 1: ramka z zer wokół obrazu sprawia, że filtr 3x3 daje wyjście tego samego rozmiaru co wejście. Liczbę kanałów zmienia tylko zwykła warstwa `PlainConv2d`.

In [ ]:
class PlainConv2d(nn.Module):  # zwykła warstwa y = H(x), zmienia liczbę kanałów
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.H = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        nn.init.kaiming_normal_(self.H.weight, nonlinearity="relu")
        nn.init.zeros_(self.H.bias)

    def forward(self, x):
        return F.relu(self.H(x))


class HighwayConv2d(nn.Module):
    def __init__(self, channels, gate_bias):
        super().__init__()
        # filtr 3x3 jak w Fitnet 4, na której paper opiera sieć; padding=1 zachowuje rozmiar obrazu
        self.H = nn.Conv2d(channels, channels, 3, padding=1)  # W_H, b_H
        self.T = nn.Conv2d(channels, channels, 3, padding=1)  # W_T, b_T
        # inicjalizacja He (jak w paperze dla sieci na MNIST): wagi z rozkładu normalnego o wariancji 2/n
        # (n = liczba wejść neuronu); przy ReLU sygnał ma taką samą wariancję w kolejnych warstwach
        nn.init.kaiming_normal_(self.H.weight, nonlinearity="relu")
        nn.init.kaiming_normal_(self.T.weight, nonlinearity="relu")
        nn.init.zeros_(self.H.bias)
        # ujemny b_T: na starcie T bliskie 0, więc warstwy głównie przepuszczają wejście
        nn.init.constant_(self.T.bias, gate_bias)

    def forward(self, x):
        h = F.relu(self.H(x))
        t = torch.sigmoid(self.T(x))
        return h * t + x * (1 - t)